In [ ]:
# ─── Cell 1: Imports & Paths ─────────────────────────────────────────────────
import os, json, pickle, dill
import numpy as np, pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise    import cosine_similarity
import gensim
from gensim import corpora
from gensim.matutils import hellinger
from gensim.utils import simple_preprocess

# paths (adjust if needed)
PAGES_PICKLE       = "../data/pages_df.pkl"
TFIDF_MATRIX_PKL   = "../data/tfidf_matrix.pkl"
TFIDF_VECT_PKL     = "../data/tfidf_vectorizer.pkl"
EMB_NPY            = "../data/page_embeddings.npy"
LDA_DICT_PKL       = "../data/lda_dictionary.pkl"
LDA_MODEL_PKL      = "../data/lda_model.pkl"
LDA_THETA_NPY      = "../data/lda_page_theta.npy"
TEST_JSON          = "../data/test_data.json"

In [ ]:
# ─── Cell 2: Load pages_df + TF–IDF artifacts ────────────────────────────────
pages_df = pd.read_pickle(PAGES_PICKLE)
with open(TFIDF_MATRIX_PKL, "rb") as f: tfidf_matrix = pickle.load(f)
with open(TFIDF_VECT_PKL,   "rb") as f: tfidf_vec    = pickle.load(f)
print(f"✅ Loaded {len(pages_df)} pages; TF–IDF matrix = {tfidf_matrix.shape}")

✅ Loaded 176 pages; TF–IDF matrix = (176, 13176)


In [ ]:
# ─── Cell 3: SBERT Embeddings Retriever (optional) ───────────────────────────
try:
    from sentence_transformers import SentenceTransformer

    if os.path.exists(EMB_NPY):
        page_embs = np.load(EMB_NPY)
    else:
        sbert = SentenceTransformer("all-mpnet-base-v2")
        page_embs = sbert.encode(pages_df.text.tolist(), show_progress_bar=True)
        np.save(EMB_NPY, page_embs)

    sbert = SentenceTransformer("all-mpnet-base-v2")
    def embed_retrieve(query, top_k=5):
        q_emb = sbert.encode([query])
        sims  = cosine_similarity(q_emb, page_embs)[0]
        idx   = np.argsort(sims)[::-1][:top_k]
        return pd.DataFrame({
            "page_name": pages_df.page_name.values[idx],
            "score":      sims[idx]
        })

    print("✅ SBERT embeddings loaded; embed_retrieve ready")
except Exception as e:
    embed_retrieve = None
    print("⚠️  Skipping SBERT retriever:", e)

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

✅ SBERT embeddings loaded; embed_retrieve ready


In [ ]:
# ─── Cell 4: TF–IDF Retriever ─────────────────────────────────────────────────
def tfidf_retrieve(query, top_k=5):
    q_vec  = tfidf_vec.transform([query])
    scores = (tfidf_matrix @ q_vec.T).toarray().ravel()
    idx    = np.argsort(scores)[::-1][:top_k]
    return pd.DataFrame({
        "page_name": pages_df.page_name.values[idx],
        "score":      scores[idx]
    })

In [ ]:
# ─── Cell 5: Gensim LDA Retriever ─────────────────────────────────────────────
texts = [simple_preprocess(txt) for txt in pages_df.text]

if os.path.exists(LDA_DICT_PKL):
    lda_dict  = corpora.Dictionary.load(LDA_DICT_PKL)
    lda_model = gensim.models.LdaModel.load(LDA_MODEL_PKL)
else:
    stop = gensim.parsing.preprocessing.STOPWORDS
    texts = [[t for t in doc if t not in stop] for doc in texts]
    lda_dict = corpora.Dictionary(texts)
    lda_dict.filter_extremes(no_below=2, no_above=0.8)
    lda_model = gensim.models.LdaModel(
        [lda_dict.doc2bow(doc) for doc in texts],
        num_topics=30, passes=10
    )
    lda_dict.save(LDA_DICT_PKL)
    lda_model.save(LDA_MODEL_PKL)

if os.path.exists(LDA_THETA_NPY):
    page_theta = np.load(LDA_THETA_NPY)
else:
    stop = gensim.parsing.preprocessing.STOPWORDS
    page_theta = np.vstack([
        np.array([p for _,p in lda_model.get_document_topics(
            lda_dict.doc2bow([t for t in simple_preprocess(txt) if t not in stop]),
            minimum_probability=0
        )])
        for txt in pages_df.text
    ])
    np.save(LDA_THETA_NPY, page_theta)

def lda_retrieve(query, top_k=5):
    stop = gensim.parsing.preprocessing.STOPWORDS
    bow = lda_dict.doc2bow([t for t in simple_preprocess(query) if t not in stop])
    q_theta = np.array([p for _,p in lda_model.get_document_topics(bow, minimum_probability=0)])
    dists = np.array([hellinger(q_theta, pt) for pt in page_theta])
    idx   = np.argsort(dists)[:top_k]
    return pd.DataFrame({
        "page_name": pages_df.page_name.values[idx],
        "score":    -dists[idx]
    })

In [ ]:
# ─── Cell 6: Save all retrievers ───────────────────────────────────────────────
with open("../data/tfidf_retriever.dill",  "wb") as f: dill.dump(tfidf_retrieve, f)
if embed_retrieve:
    with open("../data/embed_retriever.dill","wb") as f: dill.dump(embed_retrieve, f)
with open("../data/lda_retriever.dill",   "wb") as f: dill.dump(lda_retrieve, f)
print("✅ Retrievers saved:", "TF–IDF",
      ("SBERT" if embed_retrieve else "(no SBERT)"), "LDA")

✅ Retrievers saved: TF–IDF SBERT LDA


In [ ]:
# ─── Cell 7: Full Test‐Set Evaluation with explicit true_page & error analysis ─
import json

def evaluate(retriever, name):
    test = json.load(open(TEST_JSON))
    R1=R3=R5=MRR=0.0
    n=0
    failures = []  # collect misses

    for item in test:
        q         = item["query"]
        gold_page = item["true_page"]

        # run retriever
        preds = list(retriever(q, top_k=5).page_name)

        # accumulate metrics
        if gold_page in preds[:1]: R1  += 1
        if gold_page in preds[:3]: R3  += 1
        if gold_page in preds[:5]: R5  += 1
        if gold_page in preds:
            rank = preds.index(gold_page) + 1
            MRR += 1.0/rank
        else:
            # record misses
            failures.append({
                "query": q,
                "gold":  gold_page,
                "preds": preds
            })

        n += 1

    # print metrics
    print(f"\n{name:6} | R@1 {R1/n:.3f}  R@3 {R3/n:.3f}  R@5 {R5/n:.3f}  MRR {MRR/n:.3f}")

    # dump failure cases
    if failures:
        print(f"\n❗ {len(failures)} queries where {name} did NOT retrieve the gold page in top-5:")
        for f in failures:
            print("  • Query:", f["query"])
            print("    Gold:",  f["gold"])
            print("    Preds:", ", ".join(f["preds"]))
        print()

# — run all three retrievers —
print("\n— Full‐set metrics —")
evaluate(tfidf_retrieve, "TF–IDF")
if embed_retrieve:
    evaluate(embed_retrieve, "SBERT")
evaluate(lda_retrieve,   "LDA")



— Full‐set metrics —

TF–IDF | R@1 0.786  R@3 1.000  R@5 1.000  MRR 0.881

SBERT  | R@1 0.643  R@3 1.000  R@5 1.000  MRR 0.786

LDA    | R@1 0.143  R@3 0.214  R@5 0.286  MRR 0.181

❗ 10 misses for LDA:
 • Q: 'What is the maximum annual printing credit?'
   Gold: 'Printing, scanning and photocopying'
   Preds: ['Self-sourced placements', 'About the Placements Administration Team', 'About us', 'Travel and Dual Accommodation Expenses (NHS Learning Support Fund) Guidance', 'Travel and Accommodation Guidance']

 • Q: 'What is the excess for emergency medical costs?'
   Gold: 'Student Travel Insurance'
   Preds: ['Share whiteboards and documents using your camera in Teams (Microsoft update)', 'IT Services - support over the Easter period', 'Travel authorisation and booking', 'Mandatory Training (Previously known as MPPPT)', 'Your IT account']

 • Q: 'How do I apply for a coursework extension?'
   Gold: 'Consideration of Personal Circumstances'
   Preds: ['Academic Integrity and Academic Mis